# 온라인 쇼핑몰 데이터 분석 실습

대상: Python의 변수·조건문·함수를 알고 pandas를 배우기 시작한 학습자. 예상 시간: 2~3시간.
실제 개인정보나 거래 기록이 없는 합성 데이터입니다. 난수 시드는 42이며, 연말 주문 비중과 배송 기간에 따른 평점 차이를 의도적으로 반영했습니다. 현실의 시장 특성을 입증하는 데이터는 아닙니다.

## 실행 방법
ZIP을 풀고 `shop_orders.csv`와 두 노트북을 같은 폴더에 둡니다. Jupyter에서 해당 폴더를 열고 `practice.ipynb`를 실행하세요. Colab에서는 노트북을 연 뒤 파일 패널에 CSV를 업로드하세요. 필요한 패키지는 `pip install pandas numpy matplotlib jupyter`로 설치할 수 있습니다. 노트북 셀은 위에서부터 실행합니다. 정답이 궁금할 때만 `solutions.ipynb`를 여세요.

## 데이터 사전
원본은 헤더 제외 1,212행, 12열이며 중복을 제외하면 1,200개 주문입니다. 기간은 2025년 1~12월입니다. 한 행은 한 종류의 상품으로 구성된 주문 한 건입니다. 같은 고객의 여러 주문은 정상입니다.

| 열 | 의미 / 단위 |
|---|---|
| order_id | 주문 식별자. 중복 제거 기준 |
| order_date | 주문 날짜, YYYY-MM-DD |
| customer_id | 가상 고객 식별자 |
| category | Electronics 전자기기, Fashion 패션, Home 생활용품, Food 식품, Books 도서 |
| region | Seoul 서울, Busan 부산, Incheon 인천, Daegu 대구, Daejeon 대전 |
| channel | Web 웹, App 앱 |
| quantity | 주문 수량, 개 |
| unit_price | 할인 전 개당 가격, 원 |
| discount_rate | 할인율, 0.1은 10% |
| returned | 전체 반품 1, 미반품 0 |
| delivery_days | 배송 소요일, 일 |
| rating | 고객 평점, 1~5점. 빈 값은 미응답 |

## 공통 분석 규칙
1. 중복 주문을 먼저 제거하고 최초 행을 유지합니다.
2. 수량 또는 가격이 0 이하인 주문은 제외합니다. 나머지 극단값은 자동 삭제하지 않습니다.
3. 지역 결측값은 `Unknown`으로 채웁니다. 배송일 결측값은 정제 후 전체 중앙값으로 채웁니다. 평점은 결측 상태를 유지하고 평균 계산에서 제외합니다.
4. 주문 결제액 `order_amount = quantity * unit_price * (1-discount_rate)`.
5. 실습상 순매출 `net_revenue`는 미반품 주문의 결제액이며 전체 반품이면 0원입니다. 반품 시점을 별도로 기록하지 않아 원주문 월에 반영합니다. 세금·배송비·원가·부분 반품은 다루지 않습니다. 순매출은 이익이 아닙니다.
6. 객단가는 **미반품 주문 결제액 합 / 미반품 주문 수**로 정의합니다. 반품률은 **반품 주문 수 / 전체 유효 주문 수**입니다.
7. 이후 문제는 동일한 정제 데이터 `clean`을 사용합니다. 그래프에는 제목·축 이름·단위를 표시합니다. 그래프의 영어 표기는 한글 폰트 설치 없이 실행하기 위한 것입니다.

## 예시 풀이
동일한 정제 기준을 사용하면 검산값을 재현할 수 있습니다. 해석에는 여러 타당한 답이 있습니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. 데이터 구조 확인 · 기초

CSV를 읽고 상위 5행, 행·열 수, 자료형, 수치형 요약 통계, 열별 결측 개수, 중복 주문 수를 출력하세요. 날짜를 datetime으로 변환하세요.

힌트: `read_csv, head, shape, info, describe, isna, duplicated`

In [ ]:
raw = pd.read_csv("shop_orders.csv", encoding="utf-8-sig")
raw["order_date"] = pd.to_datetime(raw["order_date"])
print(raw.head())
print(raw.shape)
raw.info()
print(raw.describe())
print(raw.isna().sum())
print("중복 주문 수:", raw.duplicated("order_id").sum())

## 2. 데이터 정제 · 기초

공통 규칙에 따라 clean을 만드세요. 중복 제거 전후, 유효성 검사 전후의 행 수를 비교하고 남은 결측값을 확인하세요. 평점을 0으로 채우면 어떤 문제가 생길까요?

힌트: `drop_duplicates, loc, copy, fillna, median`

In [ ]:
dedup = raw.drop_duplicates("order_id").copy()
clean = dedup.loc[(dedup["quantity"] > 0) & (dedup["unit_price"] > 0)].copy()
clean["region"] = clean["region"].fillna("Unknown")
clean["delivery_days"] = clean["delivery_days"].fillna(clean["delivery_days"].median())
print("원본 / 중복 제거 / 유효 주문:", len(raw), len(dedup), len(clean))
print(clean.isna().sum())
# 평점 미응답을 0점으로 바꾸면 평균이 부당하게 낮아진다.

## 3. 파생 변수와 핵심 지표 · 기초

order_amount, net_revenue, month, is_weekend를 만드세요. np.where를 사용하세요. 총 순매출·반품률·미반품 객단가를 계산하세요.

힌트: `np.where, dt.month, dt.dayofweek`

In [ ]:
clean["order_amount"] = clean["quantity"] * clean["unit_price"] * (1-clean["discount_rate"])
clean["net_revenue"] = np.where(clean["returned"].eq(1), 0, clean["order_amount"])
clean["month"] = clean["order_date"].dt.month
clean["is_weekend"] = clean["order_date"].dt.dayofweek >= 5
print("총 순매출(원):", clean["net_revenue"].sum())
print("반품률:", clean["returned"].mean())
print("미반품 객단가(원):", clean.loc[clean["returned"].eq(0), "order_amount"].mean())

## 4. 조건 검색 · 기초

App으로 접수한 미반품 주문 중 결제액이 100,000원 이상인 주문을 찾고 결제액 내림차순 상위 10건을 출력하세요.

힌트: `불리언 인덱싱, sort_values`

In [ ]:
selected = clean.loc[clean["channel"].eq("App") & clean["returned"].eq(0) & clean["order_amount"].ge(100000)]
print(selected.sort_values("order_amount", ascending=False).head(10))

## 5. 카테고리 성과 · 기초

카테고리별 주문 수·순매출·반품률·평점 평균·평점 응답 수를 집계하고 순매출 내림차순 표와 막대그래프를 만드세요.

힌트: `groupby, agg, plt.bar`

In [ ]:
category = clean.groupby("category").agg(orders=("order_id","size"), revenue=("net_revenue","sum"), return_rate=("returned","mean"), rating_mean=("rating","mean"), rating_count=("rating","count")).sort_values("revenue", ascending=False)
print(category)
plt.figure(figsize=(8,4))
plt.bar(category.index, category["revenue"]/1e6)
plt.title("Net revenue by category")
plt.xlabel("Category"); plt.ylabel("Net revenue (million KRW)")
plt.tight_layout(); plt.show()

## 6. 월별 추이 · 중급

1~12월 월별 순매출과 전월 대비 증감률(%)을 계산하세요. 월 순매출과 3개월 이동평균을 한 선그래프에 표시하세요. 첫 달 증감률과 첫 두 달 이동평균이 NaN인 이유를 설명하세요.

힌트: `reindex, pct_change, rolling, plt.plot`

In [ ]:
monthly = clean.groupby("month")["net_revenue"].sum().reindex(range(1,13), fill_value=0)
growth = monthly.pct_change(fill_method=None)*100
moving = monthly.rolling(3).mean()
print(pd.DataFrame({"net_revenue":monthly,"growth_pct":growth,"moving_3m":moving}))
plt.figure(figsize=(8,4))
plt.plot(monthly.index, monthly/1e6, marker="o", label="Monthly")
plt.plot(moving.index, moving/1e6, label="3-month mean")
plt.xticks(range(1,13)); plt.xlabel("Month"); plt.ylabel("Net revenue (million KRW)")
plt.title("Monthly net revenue"); plt.legend(); plt.tight_layout(); plt.show()
# 첫 달은 이전 달이 없고, 첫 두 달은 이동평균에 필요한 3개월이 쌓이지 않았다.

## 7. 지역과 채널 비교 · 중급

행=지역, 열=채널인 순매출 피벗 테이블을 만드세요. np.divide로 지역별 App 매출 비중을 구하세요. 합계가 0인 경우 NaN을 반환하도록 처리하세요.

힌트: `pivot_table, np.divide`

In [ ]:
pivot = clean.pivot_table(index="region",columns="channel",values="net_revenue",aggfunc="sum",fill_value=0)
total = pivot.sum(axis=1).to_numpy(dtype=float)
pivot["app_share"] = np.divide(pivot["App"].to_numpy(), total, out=np.full(len(total), np.nan), where=total!=0)
print(pivot)

## 8. NumPy 기술통계 · 중급

미반품 주문 결제액을 NumPy 배열로 변환해 평균·중앙값·표본 표준편차·25/75/95백분위수를 계산하고 히스토그램을 그리세요. pandas의 std 기본값과 맞추려면 ddof를 무엇으로 지정할까요?

힌트: `to_numpy, np.mean, np.median, np.std, np.percentile, plt.hist`

In [ ]:
amounts = clean.loc[clean["returned"].eq(0),"order_amount"].to_numpy()
print("평균 / 중앙값 / 표본 표준편차:", np.mean(amounts), np.median(amounts), np.std(amounts,ddof=1))
print("25/75/95백분위수:", np.percentile(amounts,[25,75,95]))
plt.figure(figsize=(8,4)); plt.hist(amounts/1000,bins=35,edgecolor="white")
plt.xlabel("Order amount (thousand KRW)"); plt.ylabel("Order count")
plt.title("Non-returned order amounts"); plt.tight_layout(); plt.show()
# pandas Series.std의 기본 ddof=1은 표본 표준편차이다.

## 9. 이상치 탐색 · 중급

미반품 결제액에 IQR 규칙(Q1−1.5×IQR, Q3+1.5×IQR)을 적용하여 이상치 개수와 경계를 구하세요. 전체 평균과 이상치 제외 평균을 비교하세요. 고액 주문을 곧바로 오류로 보면 안 되는 이유를 설명하세요.

힌트: `np.percentile, 배열 마스크`

In [ ]:
q1,q3 = np.percentile(amounts,[25,75])
iqr = q3-q1
lo,hi = q1-1.5*iqr,q3+1.5*iqr
outliers = (amounts<lo)|(amounts>hi)
print("경계:",lo,hi,"탐지 수:",outliers.sum())
print("전체 / 제외 평균:",amounts.mean(),amounts[~outliers].mean())
# 카테고리별 가격 차이와 정상 대량 구매도 극단값을 만든다. 탐지 결과를 자동 삭제에 쓰지 않는다.

## 10. 배송과 평점 · 중급

평점 응답 주문만 골라 배송 소요일과 평점의 Pearson 상관계수를 np.corrcoef로 구하세요. 산점도를 그리고 배송일별 평균 평점·응답 수를 집계하세요. 상관관계만으로 인과관계를 주장할 수 있을까요?

힌트: `dropna, np.corrcoef, plt.scatter`

In [ ]:
rated = clean.dropna(subset=["rating"])
print("상관계수:",np.corrcoef(rated["delivery_days"],rated["rating"])[0,1])
print(rated.groupby("delivery_days")["rating"].agg(["mean","count"]))
plt.figure(figsize=(7,4)); plt.scatter(rated["delivery_days"],rated["rating"],alpha=.12)
plt.xlabel("Delivery days (missing values imputed)"); plt.ylabel("Rating")
plt.title("Delivery days and rating"); plt.tight_layout(); plt.show()
# 관측 데이터의 상관만으로 인과를 확정할 수 없다. 여기서는 생성 규칙도 관계를 포함한다.
# 중앙값 대체와 미응답 제외가 결과에 미치는 영향도 고려해야 한다.

## 11. 재구매 고객 · 중급

미반품 주문만 사용하여 고객별 주문 수와 결제액 합계를 계산하세요. 이 기간에 2회 이상 구매한 고객 비율과 결제액 상위 10명을 구하세요. 이 비율이 다음 해 재구매율과 다른 이유를 설명하세요.

힌트: `groupby, agg, ge, mean`

In [ ]:
customers = clean.loc[clean["returned"].eq(0)].groupby("customer_id").agg(orders=("order_id","size"),spend=("order_amount","sum"))
print("관측 기간 내 재구매 고객 비율:",customers["orders"].ge(2).mean())
print(customers.sort_values("spend",ascending=False).head(10))
# 분모는 기간 내 미반품 구매 고객이다. 다음 해의 구매 여부는 이 데이터에 없다.

## 12. 부트스트랩과 해석 · 도전

난수 시드 42로 미반품 주문 결제액을 원래 표본 크기만큼 복원 추출하는 과정을 2,000번 반복하세요. 평균의 부트스트랩 분포와 2.5/97.5백분위수 구간을 구하세요. 결과를 근거로 관찰 2개와 추가 분석 제안 1개를 작성하세요.

힌트: `np.random.default_rng, choice, np.percentile`

In [ ]:
boot_rng = np.random.default_rng(42)
boot_means = np.array([boot_rng.choice(amounts,size=len(amounts),replace=True).mean() for _ in range(2000)])
ci = np.percentile(boot_means,[2.5,97.5])
print("주문별 독립성을 가정한 평균의 95% 백분위 부트스트랩 구간:",ci)
plt.figure(figsize=(8,4)); plt.hist(boot_means/1000,bins=40,edgecolor="white")
for value in ci: plt.axvline(value/1000,color="red",linestyle="--")
plt.xlabel("Resampled mean amount (thousand KRW)"); plt.ylabel("Count")
plt.title("Bootstrap means (2,000 resamples)"); plt.tight_layout(); plt.show()
print("관찰: 순매출 1위 카테고리는",category.index[0],"이며, 월 순매출 최대는",monthly.idxmax(),"월이다.")
print("추가 분석: 카테고리별로 나누어 배송과 평점의 관계를 비교한다.")
# 같은 고객의 반복 주문이 있어 주문별 독립성 가정은 제한적이다.
# 심화: 고객 단위 군집 부트스트랩과 비교한다. 이 구간을 실제 시장에 일반화하지 않는다.